In [ ]:
!pip install transformers accelerate torch safetensors numpy scipy openai

!pip install -U bitsandbytes

from huggingface_hub import login
login("HF_TOKEN")
from transformers import AutoTokenizer, AutoModelForCausalLM

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.

In [ ]:
#Qwen 2.5 7B Instruct

import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch
from huggingface_hub import login

# login(token=os.getenv("HF_TOKEN"))

SIM_XLSX_PATH = "/path/to/simulations"
OUTPUT_FILE = "generated_activities_llama3_8b.xlsx"
 
PROMPT_LEVELS = {
    "Level 1": "Generate a list of 10 distinct student activities based on the provided questions, relationships, and variables so that the activities follow a natural learning order to incrementally develop understanding.",
    "Level 2": "Using the provided questions, relationships, and variables, generate a structured list of 10 distinct student activities. The activities should select one key variable to vary while keeping the others constant and students should record outputs for each variation. The activities should use outcomes from earlier activities as input for later ones so that they follow a natural learning order to incrementally develop understanding.",
    "Level 3": "Based on the provided questions, relationships, and variables, generate a detailed list of 10 distinct student activities by following these steps:\nSelect one variable to vary while keeping others constant and record the corresponding outputs.\nActivities should use outcomes from earlier activities as input for later ones.\nGroup activities that build upon each other based on overlapping variables or outputs.\nProvide clear instructions emphasizing data collection and recording observations.\nActivities should follow a natural learning order to incrementally develop understanding.",
    "Level 4": "Based on the provided questions, relationships, and variables, generate a detailed list of 10 distinct student activities by following these steps:\nSelect one variable to vary while keeping others constant and record the corresponding outputs.\nActivities should use outcomes from earlier activities as input for later ones.\nGroup activities that build upon each other based on overlapping variables or outputs.\nProvide clear instructions emphasizing data collection and recording observations.\nActivities should follow a natural learning order to incrementally develop understanding. For example, given the projectile simulation, here are a few questions: \nExplain how the launch angle affects the range of the projectile. \nWhich of the following factors can affect the range of the projectile? \nDoes increasing the initial velocity of a projectile always increase its maximum height? \nBased on these questions, here are some probable activities: \nExperiment with varying the initial velocity while keeping the launch angle constant. Record your observations on how it affected the range and time of flight. \nObserve and describe the impact of acceleration due to gravity on the time of flight. \nCompare and contrast the trajectory of a projectile at different launch angles with the same initial velocity."
}

SELECTED_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)
 
tokenizer = AutoTokenizer.from_pretrained(
    SELECTED_MODEL,
    use_auth_token=True,
    trust_remote_code=True,
    padding_side="left"
)
model = AutoModelForCausalLM.from_pretrained(
    SELECTED_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
 
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

df = pd.read_excel(SIM_XLSX_PATH)

for level_name, level_prompt in PROMPT_LEVELS.items():
    column = f"{SELECTED_MODEL.split('/')[-1]} - {level_name}"
    df[column] = ""
 
    for idx, row in df.iterrows():
        prompt_text = (
            f"{level_prompt}\n\n"
            f"Questions: {row['Questions']}\n"
            f"Relationships: {row['Relationships']}\n"
            f"Variables: {row['Variables']}\n"
        )
        output = generator(
            prompt_text,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7
        )[0]["generated_text"]
        df.at[idx, column] = output.strip()

df.to_excel(OUTPUT_FILE, index=False)
print(f"✅ Activity generation complete for {SELECTED_MODEL}. Saved to '{OUTPUT_FILE}'")